In [ ]:
0.81# This notebook applies the following basic Machine Learning models:
# Logistic Regression, SVM, KNN and Decision Trees
###

# 0. Preparation
###
# Importing libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from sklearn.metrics import classification_report, f1_score
from sklearn import linear_model, preprocessing
from sklearn.model_selection import train_test_split
from sklearn import svm, neighbors

In [ ]:
# Mounting GoogleDrive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Reading data file from GoogleDrive
df = pd.read_pickle("/content/drive/My Drive/Data Science/Team Project X-Rays/Dataframes/df_basic_1.2.pkl")
df.head()

# Define data name
df_name = "Baseline 1.2"

# Define random subsample for computation efficiency
#df = df.sample(500)


In [ ]:
# Explore Data
###

# Check data type is DataFrame
print(type(df))

# Show dimensions
print(df.shape)

# Show labels
print(df.Case.value_counts())

# Check distributions after normalisation
df.describe()

# We see that after minmax, the mean and std. deviations are still quite different


<class 'pandas.core.frame.DataFrame'>
(21105, 4098)
Case
Normal             10191
Lung_Opacity        6012
COVID               3564
Viral Pneumonia     1338
Name: count, dtype: int64


,PX_1,PX_2,PX_3,PX_4,PX_5,PX_6,PX_7,PX_8,PX_9,PX_10,...,PX_4087,PX_4088,PX_4089,PX_4090,PX_4091,PX_4092,PX_4093,PX_4094,PX_4095,PX_4096
count,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,...,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000
mean,0.174782,0.136202,0.135377,0.140458,0.145238,0.150442,0.154802,0.158104,0.161926,0.165715,...,0.632260,0.588009,0.537908,0.484236,0.425790,0.368474,0.317856,0.274771,0.246596,0.254785
std,0.255760,0.221036,0.217255,0.217726,0.218810,0.220535,0.223204,0.225479,0.227663,0.230192,...,0.286753,0.301920,0.315539,0.325191,0.329558,0.329124,0.322651,0.312106,0.301528,0.307278
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.004608,0.004115,0.004115,0.004762,0.005405,0.008000,0.008065,0.008065,0.008163,0.008197,...,0.510040,0.407821,0.270588,0.131356,0.063415,0.036290,0.022857,0.012448,0.008130,0.008299
50%,0.044000,0.031674,0.030043,0.032922,0.036145,0.037383,0.039370,0.039604,0.041667,0.043825,...,0.720648,0.679167,0.625000,0.555556,0.449393,0.313253,0.183673,0.104167,0.079208,0.089796
75%,0.240909,0.153465,0.157407,0.176707,0.196653,0.221154,0.234310,0.244980,0.255605,0.263780,...,0.845833,0.822917,0.798658,0.766393,0.725410,0.670940,0.605932,0.525253,0.451477,0.460526
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
# Check missing values
print(df.info())

print("Missing vars in columns:\n", df.isna().sum())
print("Number of total missing vars:", df.isna().sum().sum())
print("Number of total missing vars (% of all obs):", (df.isna().sum().sum())/(df.shape[0]*df.shape[1]))

# No missing vars. We can continue the ML modelling.

<class 'pandas.core.frame.DataFrame'>
Index: 21105 entries, 0 to 21164
Columns: 4098 entries, Name to PX_4096
dtypes: float64(4096), object(2)
memory usage: 660.0+ MB
None
Missing vars in columns:
 Name       0
Case       0
PX_1       0
PX_2       0
PX_3       0
          ..
PX_4092    0
PX_4093    0
PX_4094    0
PX_4095    0
PX_4096    0
Length: 4098, dtype: int64
Number of total missing vars: 0
Number of total missing vars (% of all obs): 0.0


In [ ]:
# 1. Data preprocessing
###

# Create categorical variable from Case
df["Case"] = df.Case.replace({"Normal": 0, "COVID": 1, "Lung_Opacity": 2, "Viral Pneumonia": 3})
df.Case.astype(int)

# Check construction
print(df.Case.value_counts())

# Split data into target and features
target = df.Case

# Features data: Drop Names and target
data = df.drop(["Name", "Case"], axis = 1)
data.head()
data.shape

# Split data into training and Test sets, save random state
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size = 0.2, random_state = 123)

Case
0    10191
2     6012
1     3564
3     1338
Name: count, dtype: int64


In [ ]:
# 2. Model 1 - Logistic Regression
###
import time
start_time = time.time()

# Instantiate Logistic regression for classification
clf1_name = "Logistic Regression"
clf1 = linear_model.LogisticRegression(solver='lbfgs', C = 1.0, max_iter = 10000)

# Train the model on training data
clf1.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf1.predict(X_test)

# Calc accuracys
clf1_score = clf1.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf1_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model1_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 1: --- %s minutes ---" % model1_time)

# Score and F1-Score
print("The score is:", clf1_score)
print("The mean F1-Score (unweighted) is:", clf1_f1)

# Show Confusion Matrix
cm1 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm1)

# Show classification report
model1_cr = classification_report(y_test, y_pred)
print(model1_cr)


Model 1: --- 42.53684056202571 minutes ---
The score is: 0.7367922293295428
The mean F1-Score (unweighted) is: 0.7300386656855473


Predicted Class,0,1,2,3
Realised Class,,,,
0,1729,134,223,17
1,169,368,140,4
2,272,107,779,7
3,14,12,12,234


              precision    recall  f1-score   support

           0       0.79      0.82      0.81      2103
           1       0.59      0.54      0.57       681
           2       0.68      0.67      0.67      1165
           3       0.89      0.86      0.88       272

    accuracy                           0.74      4221
   macro avg       0.74      0.72      0.73      4221
weighted avg       0.73      0.74      0.73      4221



In [ ]:
# 3. Model 2 - linear SVM
###
import time
start_time = time.time()

# Instantiate SVM
clf2_name = "linear SVM"
clf2 = svm.SVC(gamma = 0.01, kernel = "poly")

# Train the model on training data
clf2.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf2.predict(X_test)

# Calc accuracy
clf2_score = clf2.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf2_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model2_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 2: --- %s minutes ---" % model2_time)

# Score and F1-Score
print("The score is:", clf2_score)
print("The mean F1-Score (unweighted) is:", clf2_f1)

# Show Confusion Matrix
cm2 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm2)

# Show classification report
model2_cr = classification_report(y_test, y_pred)
print(model2_cr)

Model 2: --- 21.396210312843323 minutes ---
The score is: 0.8045486851457001
The mean F1-Score (unweighted) is: 0.8058695466534211


Predicted Class,0,1,2,3
Realised Class,,,,
0,1827,89,181,6
1,117,463,99,2
2,208,92,863,2
3,22,5,2,243


              precision    recall  f1-score   support

           0       0.84      0.87      0.85      2103
           1       0.71      0.68      0.70       681
           2       0.75      0.74      0.75      1165
           3       0.96      0.89      0.93       272

    accuracy                           0.80      4221
   macro avg       0.82      0.80      0.81      4221
weighted avg       0.80      0.80      0.80      4221



In [ ]:
# 3. Model 3 - KNN
###
from sklearn import neighbors
import time
start_time = time.time()

# Instantiate classifier
clf3_name = "KNN"
clf3 = neighbors.KNeighborsClassifier(n_neighbors = 7, metric = 'minkowski')

# Train the model on training data
clf3.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf3.predict(X_test)

# Calc accuracys
clf3_score = clf3.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf3_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model3_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 3: --- %s minutes ---" % model3_time)

# Score and F1-Score
print("The score is:", clf3_score)
print("The mean F1-Score (unweighted) is:", clf3_f1)

# Show Confusion Matrix
cm3 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm3)

# Show classification report
model3_cr = classification_report(y_test, y_pred)
print(model3_cr)

Model 3: --- 0.7336207350095113 minutes ---
The score is: 0.7661691542288557
The mean F1-Score (unweighted) is: 0.7443049582491339


Predicted Class,0,1,2,3
Realised Class,,,,
0,1863,76,156,8
1,188,334,157,2
2,254,88,821,2
3,43,4,9,216


              precision    recall  f1-score   support

           0       0.79      0.89      0.84      2103
           1       0.67      0.49      0.56       681
           2       0.72      0.70      0.71      1165
           3       0.95      0.79      0.86       272

    accuracy                           0.77      4221
   macro avg       0.78      0.72      0.74      4221
weighted avg       0.76      0.77      0.76      4221



In [ ]:
# 3. Model 4 - Decision Tree
###
from sklearn.tree import DecisionTreeClassifier
import time
start_time = time.time()

# Instantiate classifier
clf4_name = "Decision Tree"
clf4 = DecisionTreeClassifier(criterion = "entropy", max_depth = 4, random_state = 123)

# Train the model on training data
clf4.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf4.predict(X_test)

# Calc accuracys
clf4_score = clf4.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf4_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model4_time = (time.time() - start_time)/60


In [ ]:
# Show Results
###

# Modelling Time
print("Model 4: --- %s minutes ---" % model4_time)

# Score and F1-Score
print("The score is:", clf4_score)
print("The mean F1-Score (unweighted) is:", clf4_f1)

# Show Confusion Matrix
cm4 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm4)

# Show classification report
model4_cr = classification_report(y_test, y_pred)
print(model4_cr)
# Ideas: Could show most important Features here. But well, there are 4000 pixels...

Model 4: --- 1.0336228330930075 minutes ---
The score is: 0.6571902392797915
The mean F1-Score (unweighted) is: 0.5982033421481197


Predicted Class,0,1,2,3
Realised Class,,,,
0,1808,111,163,21
1,342,215,122,2
2,439,128,596,2
3,76,30,11,155


              precision    recall  f1-score   support

           0       0.68      0.86      0.76      2103
           1       0.44      0.32      0.37       681
           2       0.67      0.51      0.58      1165
           3       0.86      0.57      0.69       272

    accuracy                           0.66      4221
   macro avg       0.66      0.56      0.60      4221
weighted avg       0.65      0.66      0.64      4221

